# One-Click Master Runner & Comparison Dashboard (`MultiEMit`)
Click **"Run All"** to execute all Textual (`FEEL-IT` / `UmBERTo`) and Multimodal Dual-Encoder (`+ ViT-224`) experiments across all three tasks on `datasets/corpora/correctly_split/` and compile a **single unified comparison output**:
- `./results/all_experiments_summary.csv` (Master CSV table of all tasks & regimes)
- `./results/all_experiments_summary.xlsx` (Multi-sheet Excel workbook: `All_Experiments`, `Task1_Emotion`, `Task2_Irony`, `Task3_Relation`)
- `./results/all_experiments_full_report.md` (Full Markdown report with all tables + every per-label `classification_report`)

### Execution Order
1. `train_textual_emotion_sequential.ipynb` — Task 1A: Textual Emotion (Sequential `EMit A -> MultiEMit`)
2. `train_textual_emotion_concatenated.ipynb` — Task 1B: Textual Emotion (Concatenated `EMit + MultiEMit`)
3. `train_textual_irony.ipynb` — Task 2A: Textual Irony & Sarcasm (`IronITA -> MultiEMit`)
4. `train_textual_relation.ipynb` — Task 3A: Textual Semiotic Relation (`MultiEMit Multimodal Split`)
5. `multimodal_dual_encoder_emotion.ipynb` — Task 1C: Multimodal Emotion Dual-Encoder (`FEEL-IT + ViT-224`, evaluating both Sequential and Concatenated checkpoints)
6. `multimodal_dual_encoder_irony.ipynb` — Task 2B: Multimodal Irony Dual-Encoder (`UmBERTo + ViT-224`)
7. `multimodal_dual_encoder_relation.ipynb` — Task 3B: Multimodal Semiotic Relation Dual-Encoder (`UmBERTo + ViT-224`)


In [1]:
import pandas as pd
from IPython.display import Markdown, display
import importlib
import run_all_experiments as runner
importlib.reload(runner)

# ==============================================================================
# ONE-CLICK CONFIGURATION
# ==============================================================================
RUN_TEXTUAL_NOTEBOOKS = True      # Set False if textual checkpoints are already trained and you only want to run Multimodal
RUN_MULTIMODAL_NOTEBOOKS = True   # Set True to run all 3 Multimodal Dual-Encoder (+ ViT-224) notebooks
ONLY_AGGREGATE_EXISTING = False   # Set True to skip training/inference and immediately build the unified comparison tables
FORCE_RETRAIN_OVERRIDE = True     # True = retrain all checkpoints from scratch; False = reuse existing ./checkpoints/* if present


In [2]:
importlib.reload(runner)

if not ONLY_AGGREGATE_EXISTING:
    pipeline_to_run = []
    if RUN_TEXTUAL_NOTEBOOKS:
        pipeline_to_run.extend(runner.TEXTUAL_NOTEBOOKS)
    if RUN_MULTIMODAL_NOTEBOOKS:
        pipeline_to_run.extend(runner.MULTIMODAL_NOTEBOOKS)

    execution_records = []
    for nb_file, label in pipeline_to_run:
        rec = runner.run_notebook(nb_file, label, force_retrain_override=FORCE_RETRAIN_OVERRIDE)
        execution_records.append(rec)

    print("\n=== Pipeline Execution Status ===")
    display(pd.DataFrame(execution_records))



[RUNNING] Task 1A: Textual Emotion (Sequential: EMit A -> MultiEMit) (train_textual_emotion_sequential.ipynb)
Using device: cuda
10 Emotion Labels: ['Amore', 'Anticipazione', 'Disgusto', 'Fiducia', 'Gioia', 'Neutro', 'Paura', 'Rabbia', 'Sorpresa', 'Tristezza']
Step 1 (EMit A 80/20 Split) -> Train: 4772, Val: 1194

Map: 100%|██████████| 4772/4772 [00:00<00:00, 9160.76 examples/s] 

Map: 100%|██████████| 1194/1194 [00:00<00:00, 12932.85 examples/s]
d:\vsc_projects\MultiEMit\MultiEMit\.conda\Lib\site-packages\transformers\training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at MilaNLProc/feel-it-italian-emotion and are newly initialized because the shapes did not match:
- classifier.out_proj.weight: found shape torch.Size([4, 768]) in the checkpoint and torch.Size([10, 768

,notebook,label,status,returncode,elapsed_sec
0,train_textual_emotion_sequential.ipynb,Task 1A: Textual Emotion (Sequential: EMit A -...,OK,0,1343.0
1,train_textual_emotion_concatenated.ipynb,Task 1B: Textual Emotion (Concatenated: EMit +...,OK,0,1397.8
2,train_textual_irony.ipynb,Task 2A: Textual Irony & Sarcasm (IronITA -> M...,OK,0,1252.1
3,train_textual_relation.ipynb,Task 3A: Textual Semiotic Relation (MultiEMit ...,OK,0,331.2
4,multimodal_dual_encoder_emotion.ipynb,Task 1C: Multimodal Emotion Dual-Encoder (FEEL...,OK,0,82.6
5,multimodal_dual_encoder_irony.ipynb,Task 2B: Multimodal Irony Dual-Encoder (UmBERT...,OK,0,58.2
6,multimodal_dual_encoder_relation.ipynb,Task 3B: Multimodal Semiotic Relation Dual-Enc...,OK,0,67.2


## Unified Comparison Table (All Tasks, Modalities & Configurations)

In [ ]:
master_df = runner.aggregate_all_results()
if not master_df.empty:
    print("=== Master Comparison Table (All Experiments) ===")
    display(master_df)

    for task_title, kw in [
        ("Task 1: Emotion Detection (10-Label Multi-Label)", "Emotion"),
        ("Task 2: Irony & Sarcasm Detection (3-Class Single-Label)", "Irony"),
        ("Task 3: Semiotic Relation Classification (3-Class Single-Label)", "Relation"),
    ]:
        sub_df = master_df[master_df["Task"].str.contains(kw, case=False, na=False)].drop(columns=["Task"])
        if not sub_df.empty:
            print(f"\n=== {task_title} ===")
            display(sub_df.reset_index(drop=True))



UNIFIED RESULTS SAVED TO:
  1. CSV Table       : D:\vsc_projects\MultiEMit\MultiEMit\code\bert\results\all_experiments_summary.csv
  2. Excel Workbook  : D:\vsc_projects\MultiEMit\MultiEMit\code\bert\results\all_experiments_summary.xlsx
  3. Full MD Report  : D:\vsc_projects\MultiEMit\MultiEMit\code\bert\results\all_experiments_full_report.md
=== Master Comparison Table (All Experiments) ===


,Task,Modality,Notebook,Configuration,Micro F1,Macro F1,Weighted F1,Accuracy / Subset Acc
0,Emotion Detection (10-Label Multi-Label),Textual (EMit Validation),train_textual_emotion_sequential.ipynb,Step 1: FEEL-IT on EMit A (80/20 Val Split),0.5830,0.4879,0.5709,0.4472
1,Emotion Detection (10-Label Multi-Label),Textual (Sequential),train_textual_emotion_sequential.ipynb,Step 2: FEEL-IT Sequential (EMit A -> MultiEMi...,0.5107,0.4962,0.5103,0.2963
2,Emotion Detection (10-Label Multi-Label),Textual (Sequential),train_textual_emotion_sequential.ipynb,Step 3: FEEL-IT Sequential (EMit A -> MultiEMi...,0.5712,0.5479,0.5690,0.3259
3,Emotion Detection (10-Label Multi-Label),Textual (Concatenated),train_textual_emotion_concatenated.ipynb,Step 1: FEEL-IT on (EMit A + MultiEMit Textual...,0.5403,0.5092,0.5371,0.3210
4,Emotion Detection (10-Label Multi-Label),Textual (Concatenated -> Correlated Textual+MM...,train_textual_emotion_concatenated.ipynb,Step 2: Step 1 Model -> (MultiEMit Textual + M...,0.5530,0.5349,0.5507,0.3259
5,Emotion Detection (10-Label Multi-Label),Multimodal Split (Text-Only SGD),multimodal_dual_encoder_emotion.ipynb,Text-Only (FEEL-IT Sequential) — No Correlations,0.4156,0.3263,0.3913,0.1473
6,Emotion Detection (10-Label Multi-Label),Multimodal Dual-Encoder (Text + ViT-224),multimodal_dual_encoder_emotion.ipynb,Multimodal Dual-Encoder (FEEL-IT Sequential + ...,0.4155,0.3295,0.4200,0.2326
7,Emotion Detection (10-Label Multi-Label),Multimodal Split (Text-Only SGD),multimodal_dual_encoder_emotion.ipynb,Text-Only (FEEL-IT Sequential) — With Correlat...,0.4011,0.3496,0.4221,0.1473
8,Emotion Detection (10-Label Multi-Label),Multimodal Dual-Encoder (Text + ViT-224),multimodal_dual_encoder_emotion.ipynb,Multimodal Dual-Encoder (FEEL-IT Sequential + ...,0.4179,0.4616,0.4361,0.2403
9,Emotion Detection (10-Label Multi-Label),Multimodal Split (Text-Only SGD),multimodal_dual_encoder_emotion.ipynb,Text-Only (FEEL-IT Concatenated) — No Correlat...,0.3073,0.3033,0.3465,0.1628



=== Task 1: Emotion Detection (10-Label Multi-Label) ===


,Modality,Notebook,Configuration,Micro F1,Macro F1,Weighted F1,Accuracy / Subset Acc
0,Textual (EMit Validation),train_textual_emotion_sequential.ipynb,Step 1: FEEL-IT on EMit A (80/20 Val Split),0.5830,0.4879,0.5709,0.4472
1,Textual (Sequential),train_textual_emotion_sequential.ipynb,Step 2: FEEL-IT Sequential (EMit A -> MultiEMi...,0.5107,0.4962,0.5103,0.2963
2,Textual (Sequential),train_textual_emotion_sequential.ipynb,Step 3: FEEL-IT Sequential (EMit A -> MultiEMi...,0.5712,0.5479,0.5690,0.3259
3,Textual (Concatenated),train_textual_emotion_concatenated.ipynb,Step 1: FEEL-IT on (EMit A + MultiEMit Textual...,0.5403,0.5092,0.5371,0.3210
4,Textual (Concatenated -> Correlated Textual+MM...,train_textual_emotion_concatenated.ipynb,Step 2: Step 1 Model -> (MultiEMit Textual + M...,0.5530,0.5349,0.5507,0.3259
5,Multimodal Split (Text-Only SGD),multimodal_dual_encoder_emotion.ipynb,Text-Only (FEEL-IT Sequential) — No Correlations,0.4156,0.3263,0.3913,0.1473
6,Multimodal Dual-Encoder (Text + ViT-224),multimodal_dual_encoder_emotion.ipynb,Multimodal Dual-Encoder (FEEL-IT Sequential + ...,0.4155,0.3295,0.4200,0.2326
7,Multimodal Split (Text-Only SGD),multimodal_dual_encoder_emotion.ipynb,Text-Only (FEEL-IT Sequential) — With Correlat...,0.4011,0.3496,0.4221,0.1473
8,Multimodal Dual-Encoder (Text + ViT-224),multimodal_dual_encoder_emotion.ipynb,Multimodal Dual-Encoder (FEEL-IT Sequential + ...,0.4179,0.4616,0.4361,0.2403
9,Multimodal Split (Text-Only SGD),multimodal_dual_encoder_emotion.ipynb,Text-Only (FEEL-IT Concatenated) — No Correlat...,0.3073,0.3033,0.3465,0.1628



=== Task 2: Irony & Sarcasm Detection (3-Class Single-Label) ===


,Modality,Notebook,Configuration,Micro F1,Macro F1,Weighted F1,Accuracy / Subset Acc
0,Textual,train_textual_irony.ipynb,UmBERTo (IronITA Pre-trained -> MultiEMit No C...,0.7852,0.4054,0.7535,0.7852
1,Textual,train_textual_irony.ipynb,UmBERTo (IronITA Pre-trained -> MultiEMit With...,0.7901,0.5094,0.7648,0.7901
2,Multimodal Split (Text-Only SGD),multimodal_dual_encoder_irony.ipynb,Text-Only (UmBERTo) — No Correlations,0.6512,0.4603,0.7012,0.6512
3,Multimodal Dual-Encoder (Text + ViT-224),multimodal_dual_encoder_irony.ipynb,Multimodal Dual-Encoder (UmBERTo + ViT-224) — ...,0.7674,0.4280,0.7561,0.7674
4,Multimodal Split (Text-Only SGD),multimodal_dual_encoder_irony.ipynb,Text-Only (UmBERTo) — With Correlations,0.7597,0.4905,0.7711,0.7597
5,Multimodal Dual-Encoder (Text + ViT-224),multimodal_dual_encoder_irony.ipynb,Multimodal Dual-Encoder (UmBERTo + ViT-224) — ...,0.7984,0.4906,0.7782,0.7984



=== Task 3: Semiotic Relation Classification (3-Class Single-Label) ===


,Modality,Notebook,Configuration,Micro F1,Macro F1,Weighted F1,Accuracy / Subset Acc
0,Textual (on Multimodal Split),train_textual_relation.ipynb,UmBERTo Semiotic Relation (Text-Only Fine-Tune...,0.6512,0.4928,0.5975,0.6512
1,Textual (on Multimodal Split),train_textual_relation.ipynb,UmBERTo Semiotic Relation (Text-Only Fine-Tune...,0.6357,0.5486,0.6136,0.6357
2,Multimodal Split (Text-Only SGD),multimodal_dual_encoder_relation.ipynb,Text-Only (UmBERTo) — No Correlations,0.5969,0.5784,0.5961,0.5969
3,Multimodal Dual-Encoder (Text + ViT-224),multimodal_dual_encoder_relation.ipynb,Multimodal Dual-Encoder (UmBERTo + ViT-224) — ...,0.5969,0.5154,0.5840,0.5969
4,Multimodal Split (Text-Only SGD),multimodal_dual_encoder_relation.ipynb,Text-Only (UmBERTo) — With Correlations,0.6589,0.6384,0.6564,0.6589
5,Multimodal Dual-Encoder (Text + ViT-224),multimodal_dual_encoder_relation.ipynb,Multimodal Dual-Encoder (UmBERTo + ViT-224) — ...,0.6589,0.6373,0.6579,0.6589


: 